# Evaluate soft-vote Ensemble approach (Neural Network + XGBoost + Random Forest)

In [1]:
# imports
import json
import sys
import tempfile
from pathlib import Path

import numpy as np
import pandas as pd

# helpers
sys.path.insert(0, "../../")

from _settings import LABEL_COLS
from dataviz_helper import plot_confusion_matrices
from ensemble_helper import EnsemblePredictor
from models_helper import build_misclassification_table, confusion_matrices_for_predictions, evaluate_predictions

In [2]:
# define directories and paths
DATA_DIR = Path("../../3_data_curation_enrichement/3_9_split dataset to datasets and remove rare classes")
DATA_WITH_GUID = "../../3_data_curation_enrichement/3_5_clean_all_features/data_dropped_input_duplicates.parquet"
OUTPUT_PATH = "ensemble_soft_vote_metrics.json"
MISCLASS_OUTPUT_DIR = Path("../5_3_soft_vote_ensemble_model/misclassifications_per_project")

## 1. Predict the validation and test set with the three base models and average the per-class probabilities across the three models

In [3]:
# load ground truth and enrich the test set with project_code + guid (used by the misclassification analysis)
df_val = pd.read_parquet(DATA_DIR / "dataset_validation_rare_classes_removed.parquet")
df_test = pd.read_parquet(DATA_DIR / "dataset_test_rare_classes_removed.parquet")

# join project_code/guid onto the test set via the feature columns
src = pd.read_parquet(DATA_WITH_GUID)
feat_cols = [c for c in df_test.columns if not c.startswith("label_") and not c.startswith("minor_label_")]
src_dedup = src[feat_cols + ["project_code", "guid"]].drop_duplicates(subset=feat_cols)
df_test = df_test.merge(src_dedup, on=feat_cols, how="left")

# sanity check
assert df_test["project_code"].notna().all() and df_test["guid"].notna().all()

y_val = df_val[LABEL_COLS].reset_index(drop=True)
y_test = df_test[LABEL_COLS].reset_index(drop=True)

predictor = EnsemblePredictor()
probas_val = predictor.predict_proba(df_val)

# get hard predictions for the test set and model
with tempfile.TemporaryDirectory() as tmp:
    probas_test = predictor.predict_proba(df_test, split_name="test", out_dir=tmp)
    preds_test = {
        "rf": pd.read_parquet(Path(tmp) / "preds_rf_test.parquet"),
        "xgboost": pd.read_parquet(Path(tmp) / "preds_xgboost_test.parquet"),
        "nn": pd.read_parquet(Path(tmp) / "preds_nn_test.parquet"),
    }

# get the final soft-vote ensemble predictions for val and test with idmax (argmax across the class probabilities)
y_pred_val  = np.column_stack([probas_val[l].idxmax(axis=1).values  for l in LABEL_COLS])
y_pred_test = np.column_stack([probas_test[l].idxmax(axis=1).values for l in LABEL_COLS])

# treat the ensemble as just another "model" so the misclassification table picks it up uniformly
preds_test["ensemble"] = pd.DataFrame(y_pred_test, columns=LABEL_COLS)

print(f"\nValidation: {len(df_val):,} rows (y_pred shape {y_pred_val.shape})")
print(f"Test: {len(df_test):,} rows (y_pred shape {y_pred_test.shape}) — enriched with project_code + guid")

xgboost: 3.2.0 imported first
nn: 2.11.0 imported first
rf - loading /Users/lukasstoeckli/GitLabProjects/hslu_baa_ebkph_classifier/code/4_modeling/4_5_hyperparameter_tuning/models/best_rf_model_0.7143.pkl
xgboost - loading /Users/lukasstoeckli/GitLabProjects/hslu_baa_ebkph_classifier/code/4_modeling/4_5_hyperparameter_tuning/models/best_xgboost_model_0.7276.pkl
model xgboost - demo: 8,458 rows, features: 35
model rf - demo: 8,458 rows, features: 37
model xgboost wrote demo preds + probas
nn - loading /Users/lukasstoeckli/GitLabProjects/hslu_baa_ebkph_classifier/code/4_modeling/4_5_hyperparameter_tuning/models/best_nn_model_0.6784.pkl
nn - moved to mps
model nn - demo: 8,458 rows, features: 73
model rf wrote demo preds + probas
model nn wrote demo preds + probas
nn: 2.11.0 imported first
xgboost: 3.2.0 imported first
rf - loading /Users/lukasstoeckli/GitLabProjects/hslu_baa_ebkph_classifier/code/4_modeling/4_5_hyperparameter_tuning/models/best_rf_model_0.7143.pkl
xgboost - loading /User


Validation: 8,458 rows (y_pred shape (8458, 4))
Test: 3,549 rows (y_pred shape (3549, 4)) — enriched with project_code + guid


## 2. Validation set evaluation

In [4]:
val_metrics = evaluate_predictions(y_val, y_pred_val, LABEL_COLS)
val_metrics

,accuracy,precision,recall,mcc,f1_weighted,f1_macro
label,,,,,,
label_ifc_entity,0.8415,0.8447,0.8415,0.7712,0.8156,0.6955
label_predefined_type,0.5616,0.7887,0.5616,0.5219,0.6004,0.5150
label_is_external,0.8776,0.8787,0.8776,0.8022,0.8771,0.9011
label_load_bearing,0.8121,0.8257,0.8121,0.7258,0.8074,0.8196
mean,0.7732,0.8344,0.7732,0.7053,0.7751,0.7328


In [5]:
val_cms = confusion_matrices_for_predictions(y_val, y_pred_val, LABEL_COLS)

plot_confusion_matrices(val_cms, label="ifc_entity", ncols=1, save="confusion_matrix_ifc_entity_validation", chapter="appendix")
plot_confusion_matrices(val_cms, label="predefined_type", ncols=1, save="confusion_matrix_predefined_type_validation", chapter="appendix")
plot_confusion_matrices(val_cms, label=["is_external", "load_bearing"], ncols=2, height=400, save="confusion_matrix_is_external_load_bearing_validation", chapter="appendix")

figure saved: /Users/lukasstoeckli/GitLabProjects/hslu_baa_ebkph_classifier/chapters/7 appendix/img/confusion_matrix_ifc_entity_validation.svg


figure saved: /Users/lukasstoeckli/GitLabProjects/hslu_baa_ebkph_classifier/chapters/7 appendix/img/confusion_matrix_predefined_type_validation.svg


figure saved: /Users/lukasstoeckli/GitLabProjects/hslu_baa_ebkph_classifier/chapters/7 appendix/img/confusion_matrix_is_external_load_bearing_validation.svg


In [6]:
plot_confusion_matrices(val_cms, label="ifc_entity", ncols=1, normalize=True)
plot_confusion_matrices(val_cms, label="predefined_type", ncols=1, normalize=True)
plot_confusion_matrices(val_cms, label=["is_external", "load_bearing"], ncols=2, height=400, normalize=True)

## 3. Test set evaluation

In [7]:
test_metrics = evaluate_predictions(y_test, y_pred_test, LABEL_COLS)
test_metrics

,accuracy,precision,recall,mcc,f1_weighted,f1_macro
label,,,,,,
label_ifc_entity,0.9256,0.9346,0.9256,0.8923,0.9128,0.7703
label_predefined_type,0.7357,0.7628,0.7357,0.6734,0.7354,0.5477
label_is_external,0.8473,0.8520,0.8473,0.7695,0.8475,0.8636
label_load_bearing,0.7997,0.8437,0.7997,0.6954,0.8036,0.7655
mean,0.8271,0.8483,0.8271,0.7576,0.8248,0.7368


In [8]:
test_cms = confusion_matrices_for_predictions(y_test, y_pred_test, LABEL_COLS)

plot_confusion_matrices(test_cms, label="ifc_entity", ncols=1, save="confusion_matrix_ifc_entity_testset", chapter="evaluation")
plot_confusion_matrices(test_cms, label="predefined_type", ncols=1, save="confusion_matrix_predefined_type_testset", chapter="appendix")
plot_confusion_matrices(test_cms, label=["is_external", "load_bearing"], ncols=2, height=400, save="confusion_matrix_is_external_load_bearing_testset", chapter="appendix")

figure saved: /Users/lukasstoeckli/GitLabProjects/hslu_baa_ebkph_classifier/chapters/5 evaluation/img/confusion_matrix_ifc_entity_testset.svg


figure saved: /Users/lukasstoeckli/GitLabProjects/hslu_baa_ebkph_classifier/chapters/7 appendix/img/confusion_matrix_predefined_type_testset.svg


figure saved: /Users/lukasstoeckli/GitLabProjects/hslu_baa_ebkph_classifier/chapters/7 appendix/img/confusion_matrix_is_external_load_bearing_testset.svg


In [9]:
plot_confusion_matrices(test_cms, label="ifc_entity", ncols=1, normalize=True)
plot_confusion_matrices(test_cms, label="predefined_type", ncols=1, normalize=True)
plot_confusion_matrices(test_cms, label=["is_external", "load_bearing"], ncols=2, height=400, normalize=True)

## 4. Identify test-set misclassifications for the models RF + XGBoost and save CSVs per project

In [10]:
# build misclassification table on the test set
misclass_table, any_wrong_mask = build_misclassification_table(df_test, preds_test, LABEL_COLS)

# annotate each row with the base models whose hard pred matches the ensemble's soft-vote
base_models = ["rf", "xgboost", "nn"]
for label in LABEL_COLS:
    ens = misclass_table[f"{label}__ensemble_pred"].values
    matches = np.column_stack([misclass_table[f"{label}__{m}_pred"].values == ens for m in base_models])
    misclass_table[f"{label}__ensemble_voted_by"] = ["|".join(m for m, ok in zip(base_models, row) if ok) for row in matches]

feature_block = df_test[feat_cols].reset_index(drop=True)
full_table = pd.concat([misclass_table, feature_block], axis=1)

print(f"Total test rows: {len(full_table):,}")
print(f"Rows with at least one misclassification: {any_wrong_mask.sum():,} ({any_wrong_mask.mean()*100:.1f}%)")

Total test rows: 3,549
Rows with at least one misclassification: 2,169 (61.1%)


In [11]:
# write one csv per project_code, containing only the misclassified rows of that project
MISCLASS_OUTPUT_DIR.mkdir(exist_ok=True)
misclassified = full_table[any_wrong_mask]

summary = []
for project_code, group in misclassified.groupby("project_code"):
    path = MISCLASS_OUTPUT_DIR / f"{project_code}_misclassifications.csv"
    group.to_csv(path, index=False)
    summary.append({"project_code": project_code, "n_misclassified": len(group), "file": path.name})

summary_df = pd.DataFrame(summary).sort_values("n_misclassified", ascending=False).reset_index(drop=True)
summary_df.head(20)

,project_code,n_misclassified,file
0,LUMU,1139,LUMU_misclassifications.csv
1,ADEM,1030,ADEM_misclassifications.csv


## 5. Save the ensemble metrics to a JSON file for later use

In [12]:
# save results as json
results = {
    "validation": val_metrics.to_dict(orient="index"),
    "test": test_metrics.to_dict(orient="index"),
}

Path(OUTPUT_PATH).write_text(json.dumps(results, indent=2))
print(f"saved to: {Path(OUTPUT_PATH).resolve()}")

saved to: /Users/lukasstoeckli/GitLabProjects/hslu_baa_ebkph_classifier/code/5_evaluation/5_3_soft_vote_ensemble_model/ensemble_soft_vote_metrics.json
